# P8 – Stacking A→B: dùng tín hiệu stereotype (A) để cứu hate ngầm (B)

**Vấn đề.** Lỗi lớn nhất của hệ thống B hiện tại (recipe P3: top-3 mean + bias, macro-F1 0,6075) là **`yes_implicit` bị đoán thành `no`**. Trong nhãn thật, P(A = yes | B) là: `no` 18,5% · `implicit` 53,9% · `explicit` 80,2%. Nếu model A "thấy" stereotype trong một comment mà model B cho là `no`, có thể đó là hate ngầm bị bỏ sót.

**Cách làm (stacking):** thay vì lấy trung bình xác suất như P3, cho một **hồi quy logistic** học cách kết hợp xác suất OOF của nhiều model:
- `stack_B`   : đầu vào = log-xác suất B của 9 model ứng viên → đo riêng tác dụng của *stacking*.
- `stack_B+A` : thêm logit A của 9 model ứng viên → đo tác dụng của *tín hiệu A*.
- Làm đối xứng cho A (`stack_A`, `stack_A+B`).

**Trung thực như P3:** với mỗi fold k, hồi quy chỉ học trên OOF của 4 fold còn lại rồi dự đoán fold k. Siêu tham số **cố định trước** (C = 1, `class_weight="balanced"`, chuẩn hoá đặc trưng), không dò. Vì vậy điểm ở đây có **cùng phạm vi** với điểm P3 (development OOF cross-fit), so sánh trực tiếp được, nhưng **không phải điểm test**.

**Tiêu chí nhận (ROADMAP mục 6.1):** hơn recipe P3 ≥ 0,005 **và** thắng ≥ 4/5 fold; với B, F1 implicit phải tăng và F1 explicit không giảm quá 0,01.

Chỉ dùng CPU, không train model mới. Kết quả lưu ở `outputs/p8_stacking/`, **không** ghi vào `outputs/oof/` (để stacking không bị P3 nhặt làm ứng viên) và không sửa `all_results.csv`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sqe").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sqe.data import TASKS
from sqe.metrics import fold_scores, paired_delta
from sqe.selection import candidates
from sqe.validation import validate_aligned, validate_oof

pd.set_option("display.precision", 4)
pd.set_option("display.width", 200)
OUT = ROOT / "outputs" / "p8_stacking"
OUT.mkdir(parents=True, exist_ok=True)

NAMES, FRAMES = {}, {}
for t in "AB":
    NAMES[t], FRAMES[t], rejected = candidates(t)
    validate_aligned(FRAMES[t], TASKS[t][1])
    print(f"{t}: {len(NAMES[t])} ứng viên hợp lệ; bị loại: {[r['model'] for r in rejected]}")
assert (FRAMES["A"][0].StereoQueerEval_id.values == FRAMES["B"][0].StereoQueerEval_id.values).all()

P3 = {t: validate_oof(pd.read_csv(ROOT / "outputs" / "selection" / f"selected_{t}.csv",
                                  dtype={"y_true": str, "y_pred": str}), TASKS[t][1]) for t in "AB"}
for t in "AB":
    assert (P3[t].StereoQueerEval_id.values == FRAMES[t][0].StereoQueerEval_id.values).all()
    sc = fold_scores(P3[t], TASKS[t][1])
    print(f"Recipe P3 {t}: {sc.macro_f1.mean():.4f} ± {sc.macro_f1.std(ddof=0):.4f}")

A: 10 ứng viên hợp lệ; bị loại: ['deberta_comment']


B: 10 ứng viên hợp lệ; bị loại: ['deberta_comment']
Recipe P3 A: 0.7460 ± 0.0103
Recipe P3 B: 0.6159 ± 0.0170


## 1. Thăm dò: tín hiệu A có tách được "implicit bị bỏ sót" khỏi "no thật" không?
Lấy xác suất A = yes của recipe A (P3) cho các mẫu mà recipe B đoán `no`. Nếu ở một ngưỡng nào đó số implicit "cứu được" lớn hơn hẳn số `no` thật bị kéo nhầm, thì tín hiệu A có ích.

In [2]:
m = P3["B"].merge(P3["A"][["StereoQueerEval_id", "p_yes"]], on="StereoQueerEval_id")
pred_no = m[m.y_pred == "no"]
rows = []
for thr in [0.5, 0.6, 0.7, 0.8, 0.9]:
    hi = pred_no[pred_no.p_yes > thr]
    rows.append({"ngưỡng p_A(yes)": thr, "số mẫu B đoán no vượt ngưỡng": len(hi),
                 "…thật là implicit": int((hi.y_true == "yes_implicit").sum()),
                 "…thật là explicit": int((hi.y_true == "yes_explicit").sum()),
                 "…thật là no (sẽ bị kéo nhầm)": int((hi.y_true == "no").sum())})
print(f"B đoán no: {len(pred_no)} mẫu, trong đó implicit thật: {(pred_no.y_true == 'yes_implicit').sum()}")
probe = pd.DataFrame(rows)
probe

B đoán no: 1348 mẫu, trong đó implicit thật: 228


,ngưỡng p_A(yes),số mẫu B đoán no vượt ngưỡng,…thật là implicit,…thật là explicit,…thật là no (sẽ bị kéo nhầm)
0,0.5,255,57,34,164
1,0.6,165,38,22,105
2,0.7,82,15,11,56
3,0.8,31,6,5,20
4,0.9,1,0,0,1


## 2. Đặc trưng cho stacking
Với mỗi model ứng viên: subtask 2 lớp dùng **logit** log(p_yes / p_no); subtask 3 lớp dùng **log-xác suất** của cả 3 lớp. Xác suất được kẹp vào [1e-6, 1] trước khi lấy log.

In [3]:
def features(t):
    labels = TASKS[t][1]
    cols, names = [], []
    for n, o in zip(NAMES[t], FRAMES[t]):
        P = np.clip(o[[f"p_{l}" for l in labels]].to_numpy(), 1e-6, 1.0)
        if len(labels) == 2:
            cols.append(np.log(P[:, 1] / P[:, 0])[:, None])
            names.append(f"{t}:{n}:logit")
        else:
            cols.append(np.log(P))
            names += [f"{t}:{n}:log_p_{l}" for l in labels]
    return np.hstack(cols), names


X = {t: features(t) for t in "AB"}
for t in "AB":
    print(f"Đặc trưng {t}: {X[t][0].shape[1]} cột")

Đặc trưng A: 10 cột
Đặc trưng B: 30 cột


## 3. Stacking cross-fit theo fold

In [4]:
def crossfit_stack(Xm, t):
    labels = TASKS[t][1]
    base = FRAMES[t][0]
    y, folds = base.y_true.to_numpy(), base.fold.to_numpy()
    P = np.zeros((len(y), len(labels)))
    for k in sorted(np.unique(folds)):
        fit = folds != k
        model = make_pipeline(StandardScaler(),
                              LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000))
        model.fit(Xm[fit], y[fit])
        p = model.predict_proba(Xm[~fit])
        P[~fit] = p[:, [list(model.classes_).index(l) for l in labels]]
    out = base[["StereoQueerEval_id", "video_id", "fold", "y_true"]].copy()
    pcols = [f"p_{l}" for l in labels]
    out[pcols] = P
    out["y_pred"] = np.asarray(labels)[P.argmax(1)]
    return validate_oof(out[["StereoQueerEval_id", "video_id", "fold", "y_true", "y_pred", *pcols]], labels)


STACKS = {
    "A": {"stack_A": X["A"][0], "stack_A+B": np.hstack([X["A"][0], X["B"][0]])},
    "B": {"stack_B": X["B"][0], "stack_B+A": np.hstack([X["B"][0], X["A"][0]])},
}
RES = {t: {name: crossfit_stack(Xm, t) for name, Xm in STACKS[t].items()} for t in "AB"}
for t in "AB":
    for name, o in RES[t].items():
        o.to_csv(OUT / f"{name}_oof.csv", index=False, encoding="utf-8")
print("Đã lưu OOF stacking vào", OUT)

Đã lưu OOF stacking vào C:\Users\Public\Documents\NĂM 3\NLP\Đồ án\SemEval2027_Task8\outputs\p8_stacking


## 4. So với recipe P3 (so cặp theo fold)

In [5]:
rows, fold_rows = [], []
for t in "AB":
    labels = TASKS[t][1]
    base_sc = fold_scores(P3[t], labels)
    systems = {f"recipe P3 ({t})": P3[t], **RES[t]}
    for name, o in systems.items():
        sc = fold_scores(o, labels)
        d = paired_delta(sc, base_sc)
        rows.append({"subtask": t, "hệ thống": name, "macro_f1": sc.macro_f1.mean(), "std": sc.macro_f1.std(ddof=0),
                     "Δ so với P3": d["delta_mean"], "thắng P3 (fold)": f"{d['wins']}/5",
                     **{c: sc[c].mean() for c in sc.columns if c.startswith("f1_")}})
        fold_rows.append(sc.assign(subtask=t, system=name))
table = pd.DataFrame(rows)
table.to_csv(OUT / "p8_results.csv", index=False, encoding="utf-8-sig")
pd.concat(fold_rows).to_csv(OUT / "p8_folds.csv", index=False, encoding="utf-8-sig")
table

,subtask,hệ thống,macro_f1,std,Δ so với P3,thắng P3 (fold),f1_no,f1_yes,f1_yes_implicit,f1_yes_explicit
0,A,recipe P3 (A),0.7460,0.0103,0.0000,0/5,0.7720,0.7200,NaN,NaN
1,A,stack_A,0.7587,0.0073,0.0127,5/5,0.7803,0.7371,NaN,NaN
2,A,stack_A+B,0.7517,0.0106,0.0057,4/5,0.7746,0.7288,NaN,NaN
3,B,recipe P3 (B),0.6159,0.0170,0.0000,0/5,0.7507,NaN,0.4426,0.6543
4,B,stack_B,0.6252,0.0141,0.0093,3/5,0.7455,NaN,0.4806,0.6496
5,B,stack_B+A,0.6172,0.0219,0.0013,2/5,0.7382,NaN,0.4635,0.6498


**Tách riêng tác dụng của tín hiệu chéo:** so `stack_B+A` với `stack_B` (cùng là stacking, chỉ khác có thêm A hay không), và tương tự cho A.

In [6]:
for t, (with_x, without) in {"B": ("stack_B+A", "stack_B"), "A": ("stack_A+B", "stack_A")}.items():
    labels = TASKS[t][1]
    d = paired_delta(fold_scores(RES[t][with_x], labels), fold_scores(RES[t][without], labels))
    print(f"{with_x} − {without}: {d['delta_mean']:+.4f} (±{d['delta_std']:.4f}), thắng {d['wins']}/5 fold")

stack_B+A − stack_B: -0.0080 (±0.0128), thắng 2/5 fold
stack_A+B − stack_A: -0.0070 (±0.0083), thắng 1/5 fold


## 5. Các mẫu implicit bị bỏ sót được cứu / bị làm hỏng thế nào?

In [7]:
p3b, st = P3["B"].set_index("StereoQueerEval_id"), RES["B"]["stack_B+A"].set_index("StereoQueerEval_id")
imp = p3b.y_true == "yes_implicit"
moves = pd.crosstab(p3b.loc[imp, "y_pred"].rename("P3 đoán"), st.loc[imp[imp].index, "y_pred"].rename("stack_B+A đoán"))
print("Chỉ xét mẫu implicit thật:")
display(moves)
changed = (p3b.y_pred != st.y_pred)
fixed = changed & (st.y_pred == st.y_true)
broken = changed & (p3b.y_pred == p3b.y_true)
print(f"Dự đoán thay đổi: {changed.sum()} mẫu | sửa đúng: {fixed.sum()} | làm sai: {broken.sum()}")

Chỉ xét mẫu implicit thật:


stack_B+A đoán,no,yes_explicit,yes_implicit
P3 đoán,,,
no,155,12,61
yes_explicit,8,186,30
yes_implicit,17,26,277


Dự đoán thay đổi: 431 mẫu | sửa đúng: 165 | làm sai: 188


## 6. Kết luận theo tiêu chí đặt trước

In [8]:
verdict = []
for t in "AB":
    labels = TASKS[t][1]
    base_sc = fold_scores(P3[t], labels)
    for name, o in RES[t].items():
        sc = fold_scores(o, labels)
        d = paired_delta(sc, base_sc)
        ok = d["delta_mean"] >= 0.005 and d["wins"] >= 4
        if t == "B":
            ok = ok and sc.f1_yes_implicit.mean() > base_sc.f1_yes_implicit.mean() \
                 and sc.f1_yes_explicit.mean() >= base_sc.f1_yes_explicit.mean() - 0.01
        verdict.append({"subtask": t, "hệ thống": name, "Δ": round(d["delta_mean"], 4),
                        "thắng": f"{d['wins']}/5", "NHẬN?": "✅ nhận" if ok else "❌ không nhận"})
verdict = pd.DataFrame(verdict)
verdict.to_csv(OUT / "p8_verdict.csv", index=False, encoding="utf-8-sig")
verdict

,subtask,hệ thống,Δ,thắng,NHẬN?
0,A,stack_A,0.0127,5/5,✅ nhận
1,A,stack_A+B,0.0057,4/5,✅ nhận
2,B,stack_B,0.0093,3/5,❌ không nhận
3,B,stack_B+A,0.0013,2/5,❌ không nhận


## 7. Kết luận P8 (cập nhật 26/10/2026, lần 2 – sau khi có DeBERTa 5 fold)

Notebook đã chạy lại với **10 ứng viên/subtask** (thêm `deberta_p7_comment`). Recipe so sánh: A = recipe cũ (giữ nguyên sau P7), B = recipe mới có DeBERTa. Kết quả lần 1 (chưa có DeBERTa) lưu ở `outputs/audit/2026-10-26-p3-rerun/p8_lan1_truoc_deberta/`.

| Hệ thống | Lần 1 | **Lần 2** | Quyết định |
|---|---|---|---|
| `stack_A` | +0,0066, thắng 4/5 | **+0,0127, thắng 5/5**, std 0,0073 | ✅ **NHẬN** (đạt điều kiện chặt ≥ 0,005 và 5/5 đặt ngày 26/10) |
| `stack_A+B` | −0,0010 | +0,0057, 4/5 | không dùng: kém `stack_A` |
| `stack_B` | +0,0029, 2/5 | +0,0093, 3/5 | ❌ không nhận (chỉ 3/5 fold) |
| `stack_B+A` | +0,0010, 2/5 | +0,0013, 2/5 | ❌ giả thuyết A→B bị bác |

- **A→B vẫn bị bác** ở cả hai lần: tín hiệu A không giúp B (B đoán `no` mà A tự tin có stereotype thì phần lớn là `no` thật).
- **`stack_A` được nhận:** 0,7587 so với 0,7460, cả 5 fold đều dương (+0,004 đến +0,018), độ lệch giữa các fold còn nhỏ hơn recipe hiện tại.
- `stack_B` đáng chú ý vì F1 implicit 0,443 → 0,481, nhưng chỉ thắng 3/5 fold → chưa đủ.
- **Chưa triển khai:** recipe.json hiện chỉ biểu diễn trung bình có trọng số + bias. Muốn dùng `stack_A` cho test phải (1) thêm stacking thành chiến lược chính thức của P3 và (2) có xác suất test của cả 10 model A (P6.2). Việc này là mốc **P8.2** trong ROADMAP.

Phạm vi điểm: development OOF cross-fit, giống P3; không phải điểm test.